# PPO Multi-Modal RL Training for Trading

This notebook demonstrates training a Proximal Policy Optimization (PPO) agent using
multi-modal market data (summary, profile, raster, sequential) with CTAFlow.

## Components
- **Data**: `DeepIDMomentum` - multi-modal data container
- **Environment**: `MultiModalTradingEnv` - Gymnasium-compatible trading environment
- **Feature Extractor**: `WSPRExtractor` / `CnnLstmExtractor` / `MultiInputLstmExtractor`
- **Agent**: Stable-Baselines3 PPO

## Policy Architectures
1. **WSPR Policy**: Uses windowed LSTMs for summary/profile + current encoders for raster/seq
2. **CNN-LSTM Policy**: Simplified architecture with shared spatial LSTM
3. **Multi-Input LSTM Policy**: Cross-modal attention + aggregation LSTM

## 1. Setup Environment

In [ ]:

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import time, timedelta
import warnings
warnings.filterwarnings('ignore')

# Stable-Baselines3
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

# CTAFlow imports
from CTAFlow.models.intraday_momentum import DeepIDMomentum
from CTAFlow.models.deep_learning.rl import (
    MultiModalTradingEnv,
    make_ppo_policy,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Configuration

In [ ]:
# --- Data Configuration ---
TICKER = "LE_F"  # Example: Live Cattle Futures
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data'
FEATURES_PATH = PROJECT_ROOT / 'features' / TICKER
RESULTS_PATH = PROJECT_ROOT / 'results' / TICKER / 'rl'
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Data file paths
INTRADAY_PATH = DATA_PATH / 'intraday' / f'{TICKER}_5min.csv'
FEATURES_PATH_CSV = FEATURES_PATH / f'{TICKER}_features_930_1h_2.csv'
SEQUENTIAL_PATH = FEATURES_PATH / f'{TICKER}_session_0830_0930_64bin_vpin.parquet'
PROFILE_PATH = FEATURES_PATH / f'{TICKER}_session_0830_0930_64bin_profiles.npz'
RASTERIZED_PATH = FEATURES_PATH / 'rasterized_vpin.npz'
TARGET_PATH = FEATURES_PATH / f'{TICKER}_930_1030_target.csv'

# --- Environment Configuration ---
WINDOW_SIZE = 5  # Lookback window for summary/profile
MAX_SEQ_LEN = 200  # Max length for sequential data
TRANSACTION_COST_BPS = 1.0  # Transaction cost in basis points
REWARD_SCALE = 100.0  # Scale rewards (returns as %)

# --- Policy Configuration ---
POLICY_TYPE = "wspr"  # Options: 'wspr', 'cnn_lstm', 'multi_input_lstm'
D_MODEL = 128  # Base embedding dimension
LSTM_HIDDEN = 128  # LSTM hidden size
DROPOUT = 0.1

# --- Training Configuration ---
TOTAL_TIMESTEPS = 100_000
LEARNING_RATE = 3e-4
N_STEPS = 2048  # Steps per update
BATCH_SIZE = 64
N_EPOCHS = 10  # PPO epochs per update
GAMMA = 0.99  # Discount factor
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
ENT_COEF = 0.01  # Entropy coefficient (exploration)
VF_COEF = 0.5  # Value function coefficient

# Evaluation
EVAL_FREQ = 5000  # Evaluate every N timesteps
N_EVAL_EPISODES = 5

print("Configuration loaded.")
print(f"Policy type: {POLICY_TYPE}")
print(f"Window size: {WINDOW_SIZE}")
print(f"Total timesteps: {TOTAL_TIMESTEPS:,}")

## 3. Load Data

In [ ]:
# Check for required data files
print("Checking for required data files...")
for file_path in [INTRADAY_PATH, FEATURES_PATH_CSV, SEQUENTIAL_PATH, PROFILE_PATH, RASTERIZED_PATH]:
    if file_path.exists():
        print(f"  [FOUND] {file_path.name}")
    else:
        print(f"  [MISSING] {file_path}")

In [ ]:
# Load multi-modal data
model_data = DeepIDMomentum.from_files(
    intraday_path=str(INTRADAY_PATH),
    features_path=str(FEATURES_PATH_CSV),
    sequential_path=str(SEQUENTIAL_PATH),
    profile_path=str(PROFILE_PATH),
    rasterized_path=str(RASTERIZED_PATH),
    target_path=str(TARGET_PATH) if TARGET_PATH.exists() else None,
    target_col="target",
)

print("\nData loaded successfully!")
print(f"Available dates: {len(model_data.target_data)} trading days")

## 4. Preprocessing

In [ ]:
# Calculate target returns (used for reward)
model_data.calculate_target(
    target_time_end=time(14, 0),
    period_length=timedelta(minutes=60),
    make_clf=False  # Regression target for RL
)

# Scale features to comparable ranges
model_data.scale_summary_data(rolling_window=252)
model_data.normalize_sequential_features(scale_to_basis_points=True, scale_orderflow=True)

# Get dimensions from dims property
dims = model_data.dims

print(f"\nFeature dimensions:")
print(f"  Summary: {dims.summary_dim}")
print(f"  Sequential: {dims.seq_dim} (cols: {dims.seq_cols[:5]}...)")
print(f"  Profile: ({dims.profile_channels}, {dims.profile_bins})")
print(f"  Raster: ({dims.raster_bars}, {dims.raster_channels}, {dims.raster_bins})")

## 5. Create RL Environment

In [ ]:
# Split data for train/eval
n_dates = len(model_data.target_data)
train_end_idx = int(n_dates * 0.8)

# Create training environment
train_env = MultiModalTradingEnv(
    model_data=model_data,
    window_size=WINDOW_SIZE,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    max_seq_len=MAX_SEQ_LEN,
    reward_scale=REWARD_SCALE,
)

# Wrap in Monitor for logging
train_env = Monitor(train_env, str(RESULTS_PATH / 'train_monitor'))

# Create evaluation environment (uses same data but different for eval callback)
eval_env = MultiModalTradingEnv(
    model_data=model_data,
    window_size=WINDOW_SIZE,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    max_seq_len=MAX_SEQ_LEN,
    reward_scale=REWARD_SCALE,
)
eval_env = Monitor(eval_env, str(RESULTS_PATH / 'eval_monitor'))

print(f"\nEnvironment created:")
print(f"  Observation space: {train_env.observation_space}")
print(f"  Action space: {train_env.action_space}")
print(f"  Episode length: ~{n_dates - WINDOW_SIZE} steps")

## 6. Policy Selection

Choose one of three policy architectures:
1. **WSPR** (default): Full multi-path architecture with separate LSTMs
2. **CNN-LSTM**: Simplified with shared spatial LSTM
3. **Multi-Input LSTM**: Cross-modal attention + aggregation

In [ ]:
# Policy comparison info
policy_info = {
    'wspr': {
        'name': 'WSPR Extractor',
        'desc': 'Windowed Summary/Profile LSTMs + Current Raster/Seq encoders + Spatial fusion',
        'params_est': 'Medium (~500K)',
    },
    'cnn_lstm': {
        'name': 'CNN-LSTM Extractor',
        'desc': 'Profile/Raster CNNs -> Shared spatial LSTM + Summary/Seq encoders',
        'params_est': 'Small (~300K)',
    },
    'multi_input_lstm': {
        'name': 'Multi-Input LSTM Extractor',
        'desc': 'Per-modality encoders -> Cross-modal attention -> Aggregation LSTM',
        'params_est': 'Large (~800K)',
    },
}

print("Available Policy Architectures:")
print("=" * 60)
for key, info in policy_info.items():
    marker = " <-- SELECTED" if key == POLICY_TYPE else ""
    print(f"\n{key}:{marker}")
    print(f"  Name: {info['name']}")
    print(f"  Description: {info['desc']}")
    print(f"  Estimated params: {info['params_est']}")

In [ ]:
# Create PPO model with selected policy
model = make_ppo_policy(
    env=train_env,
    policy_type=POLICY_TYPE,
    dims=dims,
    d_model=D_MODEL,
    lstm_hidden=LSTM_HIDDEN,
    learning_rate=LEARNING_RATE,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=CLIP_RANGE,
    ent_coef=ENT_COEF,
    vf_coef=VF_COEF,
    device=str(device),
    verbose=1,
    tensorboard_log=str(RESULTS_PATH / 'tensorboard'),
    dropout=DROPOUT,
)

# Print model info
total_params = sum(p.numel() for p in model.policy.parameters() if p.requires_grad)
print(f"\nPPO Model created with {POLICY_TYPE} policy")
print(f"Total trainable parameters: {total_params:,}")

## 7. Setup Callbacks

In [ ]:
# Evaluation callback
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(RESULTS_PATH / 'best_model'),
    log_path=str(RESULTS_PATH / 'eval_logs'),
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    render=False,
)

print(f"Evaluation callback configured:")
print(f"  Eval frequency: every {EVAL_FREQ} timesteps")
print(f"  Eval episodes: {N_EVAL_EPISODES}")
print(f"  Best model save path: {RESULTS_PATH / 'best_model'}")

## 8. Training

In [ ]:
print(f"Starting training for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"Policy: {POLICY_TYPE}")
print(f"Device: {device}")
print("="*60)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=eval_callback,
    progress_bar=True,
)

print("\nTraining complete!")

## 9. Visualization: Training Progress

In [ ]:
# Load evaluation results
eval_results_path = RESULTS_PATH / 'eval_logs' / 'evaluations.npz'
if eval_results_path.exists():
    eval_data = np.load(eval_results_path)
    timesteps = eval_data['timesteps']
    results = eval_data['results']  # (n_evals, n_episodes)
    ep_lengths = eval_data['ep_lengths']
    
    mean_rewards = results.mean(axis=1)
    std_rewards = results.std(axis=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Mean reward over training
    ax1 = axes[0]
    ax1.plot(timesteps, mean_rewards, 'b-', label='Mean Reward')
    ax1.fill_between(timesteps, mean_rewards - std_rewards, mean_rewards + std_rewards, 
                     alpha=0.3, label='Std Dev')
    ax1.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Timesteps')
    ax1.set_ylabel('Episode Reward')
    ax1.set_title(f'Training Progress ({POLICY_TYPE.upper()} Policy)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Episode lengths
    ax2 = axes[1]
    ax2.plot(timesteps, ep_lengths.mean(axis=1), 'g-')
    ax2.set_xlabel('Timesteps')
    ax2.set_ylabel('Episode Length')
    ax2.set_title('Episode Lengths')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_PATH / f'{TICKER}_rl_training_progress.png', dpi=150)
    plt.show()
    
    print(f"\nTraining Summary:")
    print(f"  Final mean reward: {mean_rewards[-1]:.4f}")
    print(f"  Best mean reward: {mean_rewards.max():.4f} (at step {timesteps[mean_rewards.argmax()]:,})")
else:
    print("No evaluation results found. Run training first.")

## 10. Visualization: Action Distribution

In [ ]:
# Run evaluation episode and collect actions
obs, _ = eval_env.reset()
actions = []
rewards = []
positions = []
dates = []

done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    done = terminated or truncated
    
    actions.append(action)
    rewards.append(reward)
    positions.append(info.get('position', action - 1))
    dates.append(info.get('date', None))

actions = np.array(actions)
rewards = np.array(rewards)
positions = np.array(positions)

print(f"Evaluation episode completed: {len(actions)} steps")
print(f"Total reward: {rewards.sum():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Action distribution
ax1 = axes[0, 0]
action_labels = ['Short', 'Neutral', 'Long']
action_counts = [(actions == i).sum() for i in range(3)]
colors = ['red', 'gray', 'green']
bars = ax1.bar(action_labels, action_counts, color=colors, edgecolor='black')
ax1.set_ylabel('Count')
ax1.set_title('Action Distribution')
for bar, count in zip(bars, action_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{count}\n({100*count/len(actions):.1f}%)', ha='center', va='bottom')

# Plot 2: Cumulative reward
ax2 = axes[0, 1]
cumulative_reward = np.cumsum(rewards)
ax2.plot(cumulative_reward, 'b-', linewidth=1)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.fill_between(range(len(cumulative_reward)), 0, cumulative_reward, 
                 where=cumulative_reward >= 0, alpha=0.3, color='green')
ax2.fill_between(range(len(cumulative_reward)), 0, cumulative_reward, 
                 where=cumulative_reward < 0, alpha=0.3, color='red')
ax2.set_xlabel('Step')
ax2.set_ylabel('Cumulative Reward')
ax2.set_title('Cumulative Reward Over Episode')
ax2.grid(True, alpha=0.3)

# Plot 3: Position over time
ax3 = axes[1, 0]
ax3.plot(positions, 'k-', linewidth=0.5, alpha=0.7)
ax3.fill_between(range(len(positions)), 0, positions, 
                 where=np.array(positions) > 0, alpha=0.5, color='green', label='Long')
ax3.fill_between(range(len(positions)), 0, positions, 
                 where=np.array(positions) < 0, alpha=0.5, color='red', label='Short')
ax3.set_xlabel('Step')
ax3.set_ylabel('Position')
ax3.set_title('Position Over Time')
ax3.legend()
ax3.set_ylim(-1.5, 1.5)
ax3.grid(True, alpha=0.3)

# Plot 4: Reward distribution by action
ax4 = axes[1, 1]
reward_by_action = [rewards[actions == i] for i in range(3)]
bp = ax4.boxplot(reward_by_action, labels=action_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
ax4.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax4.set_ylabel('Reward')
ax4.set_title('Reward Distribution by Action')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH / f'{TICKER}_rl_action_analysis.png', dpi=150)
plt.show()

## 11. Performance Metrics

In [ ]:
# Calculate performance metrics
total_return = rewards.sum()
mean_reward = rewards.mean()
std_reward = rewards.std()
sharpe = mean_reward / std_reward * np.sqrt(252) if std_reward > 0 else 0

# Position metrics
long_pct = (positions == 1).mean() * 100
short_pct = (positions == -1).mean() * 100
neutral_pct = (positions == 0).mean() * 100

# Transition frequency (position changes)
transitions = np.sum(np.diff(positions) != 0)
transition_freq = transitions / len(positions) * 100

# Win rate
profitable_trades = (rewards > 0).sum()
win_rate = profitable_trades / len(rewards) * 100

print("=" * 50)
print("PERFORMANCE METRICS")
print("=" * 50)
print(f"\nReturns:")
print(f"  Total Return: {total_return:.4f}")
print(f"  Mean Daily Return: {mean_reward:.6f}")
print(f"  Std Daily Return: {std_reward:.6f}")
print(f"  Annualized Sharpe: {sharpe:.4f}")

print(f"\nPositions:")
print(f"  Long: {long_pct:.1f}%")
print(f"  Neutral: {neutral_pct:.1f}%")
print(f"  Short: {short_pct:.1f}%")

print(f"\nTrading Activity:")
print(f"  Position Changes: {transitions}")
print(f"  Transition Frequency: {transition_freq:.1f}%")
print(f"  Win Rate: {win_rate:.1f}%")
print("=" * 50)

## 12. Save Model

In [ ]:
# Save the trained model
model_save_path = RESULTS_PATH / f'{TICKER}_ppo_{POLICY_TYPE}.zip'
model.save(str(model_save_path))
print(f"Model saved to: {model_save_path}")

# Save configuration
config = {
    'ticker': TICKER,
    'policy_type': POLICY_TYPE,
    'window_size': WINDOW_SIZE,
    'd_model': D_MODEL,
    'lstm_hidden': LSTM_HIDDEN,
    'total_timesteps': TOTAL_TIMESTEPS,
    'learning_rate': LEARNING_RATE,
    'transaction_cost_bps': TRANSACTION_COST_BPS,
    'dims': {
        'summary_dim': dims.summary_dim,
        'seq_dim': dims.seq_dim,
        'profile_channels': dims.profile_channels,
        'profile_bins': dims.profile_bins,
        'raster_bars': dims.raster_bars,
        'raster_channels': dims.raster_channels,
        'raster_bins': dims.raster_bins,
    },
    'final_metrics': {
        'total_return': float(total_return),
        'sharpe': float(sharpe),
        'win_rate': float(win_rate),
    }
}

import json
config_path = RESULTS_PATH / f'{TICKER}_ppo_{POLICY_TYPE}_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Config saved to: {config_path}")

## 13. Compare Policies (Optional)

Run this section to train and compare all three policy architectures.

In [ ]:
# Set to True to run policy comparison
RUN_COMPARISON = False

if RUN_COMPARISON:
    comparison_results = {}
    
    for policy_type in ['wspr', 'cnn_lstm', 'multi_input_lstm']:
        print(f"\n{'='*60}")
        print(f"Training {policy_type.upper()} policy...")
        print(f"{'='*60}")
        
        # Create model
        policy_model = make_ppo_policy(
            env=train_env,
            policy_type=policy_type,
            dims=dims,
            d_model=D_MODEL,
            lstm_hidden=LSTM_HIDDEN,
            learning_rate=LEARNING_RATE,
            n_steps=N_STEPS,
            batch_size=BATCH_SIZE,
            n_epochs=N_EPOCHS,
            device=str(device),
            verbose=0,
        )
        
        # Train (reduced timesteps for comparison)
        policy_model.learn(total_timesteps=50_000, progress_bar=True)
        
        # Evaluate
        obs, _ = eval_env.reset()
        eval_rewards = []
        done = False
        while not done:
            action, _ = policy_model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = eval_env.step(action)
            done = terminated or truncated
            eval_rewards.append(reward)
        
        eval_rewards = np.array(eval_rewards)
        comparison_results[policy_type] = {
            'total_return': eval_rewards.sum(),
            'sharpe': eval_rewards.mean() / eval_rewards.std() * np.sqrt(252) if eval_rewards.std() > 0 else 0,
            'params': sum(p.numel() for p in policy_model.policy.parameters() if p.requires_grad),
        }
    
    # Display comparison
    print("\n" + "="*60)
    print("POLICY COMPARISON RESULTS")
    print("="*60)
    df_comparison = pd.DataFrame(comparison_results).T
    df_comparison['total_return'] = df_comparison['total_return'].round(4)
    df_comparison['sharpe'] = df_comparison['sharpe'].round(4)
    df_comparison['params'] = df_comparison['params'].apply(lambda x: f"{x:,}")
    display(df_comparison)
else:
    print("Policy comparison skipped. Set RUN_COMPARISON=True to run.")

## 14. Load and Test Saved Model

In [ ]:
# Example: Load saved model for inference
# loaded_model = PPO.load(str(model_save_path))
# 
# obs, _ = eval_env.reset()
# action, _ = loaded_model.predict(obs, deterministic=True)
# print(f"Predicted action: {['Short', 'Neutral', 'Long'][action]}")

print("To load the model later, use:")
print(f"  model = PPO.load('{model_save_path}')")
print(f"  action, _ = model.predict(observation, deterministic=True)")